# പ്രാധാന്യമുള്ള അംഗ മിഡിൽവെയർ സഹിതം ഹോട്ടൽ ബുക്കിംഗ്

മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ഉപയോഗിച്ച് **ഫങ்ஶൻ അടിസ്ഥാനമാക്കിയ മിഡിൽവെയർ** പ്രദർശിപ്പിക്കുന്ന ഈ നോട്ട്ബുക്ക് ആണ്. പ്രാധാന്യമുള്ള അംഗങ്ങൾക്ക് പ്രത്യേക അവകാശങ്ങൾ നൽകുന്ന ഒരു മിഡിൽവെയർ ലെയർ ചേർത്തുകൊണ്ട് നാം കൺഡീഷണൽ വർക്ക്‌ഫ്ലോയുടെ ഉദാഹരണം വികസിപ്പിക്കുന്നു.

## നിങ്ങൾ പഠിക്കും:
1. **ഫങ്ഷൻ-ആധാരിത മിഡിൽവെയർ**: ഫങ്ഷൻ ഫലങ്ങൾ ഇടപെടുകയും തിരുത്തുകയും ചെയ്യുക
2. **സന്ദർഭ ആക്‌സസ്**: എക്സിക്യൂഷൻ കഴിഞ്ഞ് `context.result` വായിക്കയും തിരുത്തുകയും ചെയ്യുക
3. **ബിസിനസ് ലജിക് നടപ്പാക്കൽ**: പ്രാധാന്യമുള്ള അംഗങ്ങളുടെ ഗുണങ്ങൾ
4. **ഫലം അറ്റകുറ്റപ്പണി**: ഉപയോക്തൃ നിലയിൽ അടിസ്ഥാനപ്പെടുത്തി ഫങ്ഷൻ ഫലങ്ങൾ മാറ്റുക
5. **അത്തരത്തിലൊരു വർക്ക്‌ഫ്ലോ, വ്യത്യസ്ത ഫലങ്ങൾ**: മിഡിൽവെയർ-നിർദേശിത പെരുമാറ്റ മാറ്റങ്ങൾ

## മിഡിൽവെയർ സഹിതം വർക്ക്‌ഫ്ലോ ആർക്കിടെക്ചർ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## കൺഡീഷണൽ വർക്ക്‌ഫ്ലോയിൽ നിന്നുള്ള മുഖ്യ വ്യത്യാസം:

**മിഡിൽവെയർ ഇല്ലാതെ** (14-conditional-workflow.ipynb):
- പാരീസ്‌ക്ക് മുറികൾ ഇല്ല → alternative_agent എന്നിലേക്ക് റൂട്ടുചെയ്യുക

**മിഡിൽവെയർ ഉള്ളപ്പോൾ** (ഈ നോട്ട്ബുക്ക്):
- പതിവു ഉപയോക്താവ് + പാരീസ് → മുറികൾ ഇല്ല → alternative_agent എന്നിലേക്ക് റൂട്ടുചെയ്യം
- പ്രാധാന്യമുള്ള ഉപയോക്താവ് + പാരീസ് → 🌟 മിഡിൽവെയർ ഒവറൈഡുകൾ! → ലഭ്യമാണ് → booking_agent എന്നിലേക്ക് റൂട്ടുചെയ്യുക

## മുൻപരിഗണനകൾ:
- മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ഇൻസ്റ്റാൾ ചെയ്തിരിക്കണം
- കൺഡീഷണൽ വർക്ക്‌ഫ്ലോകൾക്കുള്ള അറിവ് (14-conditional-workflow.ipynb കാണുക)
- GitHub ടോക്കൺ അല്ലെങ്കിൽ OpenAI API കീ
- മിഡിൽവെയർ പാറ്റേണുകളുടെ അടിസ്ഥാന മനസ്സിലാക്കൽ


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ഘട്ടം 1: ഘടനാത്മക ഔട്ട്പുട്ടുകൾക്കായി Pydantic മോഡലുകൾ നിർവചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകും എന്ന **സ്കീമ** നിർവചിക്കുന്നു. മിഡിൽവെയർ ലഭ്യത ഫലം മാറ്റുമ്പോൾ അതിനെ കുറിക്കാൻ `priority_override` ഫീൽഡ് ഞങ്ങൾ ചേർത്തിട്ടുണ്ട്.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ഘട്ടം 2: പ്രാധാന്യമുള്ള അംഗങ്ങളുടെ ഡാറ്റാബേസ് നിർവചിക്കുക

ഈ ഡെമോയിന്, നമ്മുക്ക് പ്രാധാന്യമുള്ള അംഗത്വ ഡാറ്റാബേസ് സിമുലേറ്റ് ചെയ്യാം. പ്രൊഡക്ഷനിൽ, ഇത് യഥാർത്ഥ ഡാറ്റാബേസ് അല്ലെങ്കിൽ API ക്വറി ചെയ്യുന്നതാകും.

**പ്രാധാന്യമുള്ള അംഗങ്ങൾ:**
- `alice@example.com` - VIP അംഗം
- `bob@example.com` - പ്രീമിയം അംഗം  
- `priority_user` - പരീക്ഷണ അക്കൗണ്ട്


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ഘട്ടം 3: ഹോട്ടൽ ബുക്കിംഗ് ടൂൾ സൃഷ്ടിക്കുക

വ്യവസ്ഥാപൂർവ്വക വർക്ക്‌ഫ്ലോവിനെപ്പോലെ തന്നെയാണ്, എന്നാൽ ഇപ്പോൾ അത് മിഡിൽവെയർ തടയാൻ പോകും!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ഘട്ടം 4: 🌟 പ്രാധാന്യമുള്ള ചെക്ക് മിഡിൽവെയർ സൃഷ്ടിക്കുക (പ്രധാന ഫീച്ചർ!)

ഇതാണ് ഈ നോട്ട്ബുക്കിന്റെ **പ്രധാന ഫംഗ്ഷണാലിറ്റി**. മിഡിൽവെയർ:

1. **ഇന്റർസെപ്റ്റ്** ചെയ്യുന്നു hotel_booking ഫംഗ്ഷൻ കോളിനെ
2. `next(context)` കാൾ ചെയ്ത് ഫംഗ്ഷൻ സാധാരണയായി **പ്രവർത്തിപ്പിക്കുന്നു**
3. ഫലം `context.result` ൽ **പരിശോധിക്കുന്നു**
4. ഉപയോക്താവ് പ്രാധാന്യമുള്ളവനായി അടയാളപ്പെടുത്തിയിരിക്കുമ്പോൾ മുറികളില്ലെങ്കിൽ ഫലം **ഓവർറൈഡ്** ചെയ്യുന്നു
5. തിരുത്തിയ ഫലം ഏജൻറിന് **തിരികെ നൽകുന്നു**

**പ്രധാന മാതൃക:**
```python
async def my_middleware(context, next):
    await next(context)  # ഫംഗ്ഷൻ നിർവ്വഹിക്കുക
    # ഇപ്പോൾ context.result-ലുണ്ട് ഫംഗ്ഷന്റെ ഔട്ട്പുട്ട്
    if some_condition:
        context.result = new_value  # വായില്‍ മുകളിൽ എഴുതുക!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## ഘട്ടം 5: റൂട്ടിംഗിനുള്ള സാഹചര്യ ഫംഗ്ഷനുകൾ നിർവ്വചിക്കുക

ഷരത്വ ഘടനയിലെ_conditional workflow_ പോലെ തന്നെയാണ് - അവ റൂട്ടിംഗ് നിർണയിക്കാൻ ഘടനാപരമായ ഔട്ട്പുട്ട് പരിശോധിക്കുന്നു.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## ഘട്ടം 6: കസ്റ്റം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

മുന്‍പ് ചെയ്തതേതുപോലെ అదే എക്സിക്യൂട്ടർ - അന്തിമ വര്‍ക്ക്ഫ്ലോ ഔട്ട്പുട്ട് പ്രദർശിപ്പിക്കുന്നു.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## ഘട്ടം 7: പരിപാലന ചേരുക്കളിൽ ലഭ്യമാക്കുക

LLM ക്ലയന്റ് (Microsoft Foundry / Azure OpenAI) ക്രമീകരിക്കുക.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## പടി 8: മിഡിൽവെയർ ഉപയോഗിച്ച് AI ഏജന്റ്സ് സൃഷ്ടിക്കുക

**പ്രധാന വ്യത്യാസം:** availability_agent സൃഷ്ടിക്കുമ്പോൾ, നാം `middleware` പാരാമീറ്റർ പായസം ചെയ്യുന്നു!

എങ്ങനെ priority_check_middleware ഏജന്റിന്റെ ഫങ്ഷൻ കോൾ പൈപ്പ്ലൈനിലേക്ക് ഇൻജെക്ട് ചെയ്യുന്നു എന്നതാണ് ഇത്.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ഘട്ടം 9: വർക്ക്ഫ്ലോ നിർമ്മിക്കുക

മുമ്പത്തെതുപോലെ തന്നെ workflow ഘടന - ലഭ്യതയെ ആശ്രയിച്ചുള്ള നിബന്ധനാത്മക റൂട്ടിംഗ്.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## പടി 10: ടെസ്റ്റ് കേസ് 1 - പാരീസിലെ സാധാരണ ഉപയോക്താവ് (ഓവർറൈഡ് ഇല്ല)

ഒരു സാധാരണ ഉപയോക്താവ് പാരീസ് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → റൂം ഇല്ല → alternative_agent ലേക്ക് റൂട്ടുചെയ്യുന്നു


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ഘട്ടം 11: ടെസ്റ്റ് കേസ് 2 - 🌟 പ്രിയതര ഉപയോഗങ്ങളുടെ പാരീസ് (ഒവരൈഡ് ഉപയോഗിച്ച്!)

ഒരു പ്രിയതര അംഗം പാരീസ് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → ആദ്യം റൂമുകൾ ലഭ്യമല്ല → 🌟 മിഡിൽവെയർ ഒവരൈഡ് ചെയ്യുന്നു! → booking_agent ലേക്ക് റൂട്ടുചെയ്യുന്നു

**ഇതാണ് മിഡിൽവെയർ ശക്തിയുടെ പ്രധാന പ്രകടനം!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ദ ഷെപ്പ് 12: ടെസ്റ്റ് കേസ് 3 - സ്റ്റോക്ക്‌ഹോംയിൽ പ്രാധാന്യമുള്ള ഉപയോക്താവ് (ഇതിനകം ലഭ്യമാണ്)

പ്രാധാന്യമുള്ള ഉപയോക്താവ് സ്റ്റോക്ക്‌ഹോം ശ്രമിക്കുന്നു → മുറികൾ ലഭ്യമാണ് → ഊർജ്ജിതമാക്കൽ ആവശ്യമില്ല → ബുക്കിംഗ് ഏജന്റിലേക്ക് റൂട്ടുചെയ്യുന്നു

ഇത് മിഡിൽവെയർ ആവശ്യത്തിന് മാത്രമേ പ്രവർത്തിക്കൂ എന്ന കാര്യം കാണിക്കുന്നു!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## പ്രധാന കാര്യങ്ങളും മിഡിൽവെയർ സങ്കൽപ്പങ്ങൾ

### ✅ നിങ്ങൾ പഠിച്ചവ:

#### **1. ഫംഗ്ഷൻ-അടിസ്ഥാനമായ മിഡിൽവെയർ മാതൃക**

മിഡിൽവെയർ ഒരു ലളിതമായ അസിങ്ക് ഫംഗ്ഷൻ ഉപയോഗിച്ച് ഫംഗ്ഷൻ കോളുകൾ തടയുന്നു:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ഫംഗ്ഷൻ പ്രവർത്തനം ആരംഭിക്കുന്നതിന് മുമ്പ്
    print("Intercepting...")
    
    # ഫംഗ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    await next(context)
    
    # ഫംഗ്ഷൻ പ്രവർത്തനം കഴിഞ്ഞ് - ഫലം പരിശോധിക്കുക
    if context.result:
        # ആവശ്യമെങ്കിൽ ഫലം മാറ്റുക
        context.result = modified_value
```

#### **2. സന്ധർഭം ആക്‌സസ് ചെയ്യൽ 및 ഫലം ഒവറൈഡ് ചെയ്യൽ**

- `context.function` - വിളിക്കുന്ന ഫംഗ്ഷൻ ആക്‌സസ് ചെയ്യുക
- `context.arguments` - ഫംഗ്ഷൻ_arguments വായിക്കുക
- `context.kwargs` - അധിക പാരാമീറ്ററുകൾ ആക്‌സസ് ചെയ്യുക
- `await next(context)` - ഫംഗ്ഷൻ അറുക്കുക
- `context.result` - ഫംഗ്ഷന്റെ ഔട്ട്പുട്ട് വായിച്ച്/മാറ്റുക

#### **3. ബിസിനസ്സിന്റെ തർക്കം നടപ്പാക്കൽ**

നമ്മുടെ മിഡിൽവെയർ മുൻഗണനഉളഞ്ഞ മെമ്പർ പ്രയോജനങ്ങൾ നടപ്പിലാക്കുന്നു:
- **രംഗീകരിച്ച ഉപയോക്താക്കൾ**: മാറ്റങ്ങൾ ഇല്ല, സാധാരണ പ്രവൃത്തി പ്രവാഹം
- **മുൻഗണനയുള്ള ഉപയോക്താക്കൾ**: "ലഭ്യമായില്ല" → "ലഭ്യമാണ്" ഒവറൈഡ് ചെയ്യുക
- **പാടിയ മാനദണ്ഡം**: ആവശ്യമുള്ളപ്പോൾ മാത്രമേ ഓവർറൈഡ് ചെയ്യൂ

#### **4. ഒരേ പ്രവൃത്തി പ്രവാഹം, വ്യത്യസ്ത ഫലങ്ങൾ**

മിഡിൽവെയറിന്റെ ശക്തി:
- ✅ പ്രവൃത്തി പ്രവാഹ ഘടനയിൽ മാറ്റങ്ങളൊന്നുമില്ല
- ✅ ടൂൾ ഫംഗ്ഷനിൽ മാറ്റങ്ങളൊന്നുമില്ല
- ✅ പാടിയ റൂട്ടിംഗ് തർക്കത്തിൽ മാറ്റങ്ങളൊന്നുമില്ല
- ✅ വെറും മിഡിൽവെയർ → വ്യത്യസ്ത പെരുമാറ്റം!

### 🚀 യഥാർത്ഥ ലോക പ്രയോഗങ്ങൾ:

1. **വിഐപി/പ്രീമിയം ഫീച്ചറുകൾ**
   - പ്രീമിയം ഉപയോക്താക്കൾക്ക് നിരക്ക് പരിധികൾ ഒവറൈഡ് ചെയ്യുക
   - ഉറവിടങ്ങൾക്ക് മുൻഗണന ആക്‌സസ് നൽകുക
   - പ്രീമിയം ഫീച്ചറുകൾ ഡൈനാമിക്കായി അൺലോക്ക് ചെയ്യുക

2. **എ/ബി ടെസ്റ്റിംഗ്**
   - ഉപയോക്താക്കളെ വ്യത്യസ്ത നടപ്പിലാക്കലുകളിലേക്ക് റൂട്ടുചെയ്യുക
   - പുതിയ ഫീച്ചറുകൾ പ്രത്യേക ഉപയോക്താക്കളുമായി പരീക്ഷിക്കുക
   - രംഗപ്രവേശകമായ ഫീച്ചർ റോളൗട്ടുകൾ

3. **സുരക്ഷയും പാലനവും**
   - ഫംഗ്ഷൻ കോളുകളും ഓഡിറ്റ് ചെയ്യുക
   - സენსിറ്റീവ് പ്രവർത്തനങ്ങൾ തടയുക
   - ബിസിനസ് നിയമങ്ങൾ നടപ്പിലാക്കുക

4. **പ്രകടന മെച്ചപ്പെടുത്തൽ**
   - പ്രത്യേക ഉപയോക്താക്കൾക്കായി ഫലം കാഷ് ചെയ്യുക
   - സാധ്യമായപ്പോൾ ചെലവേറിയ പ്രവർത്തനങ്ങൾ ഒഴിവാക്കുക
   - ഡൈനാമിക് സ്രോതസ്സ് നിയോഗനം

5. **പിഴവുകൾ കൈകാര്യം ചെയ്യലും വീണ്ടും ശ്രമിക്കൽ**
   - പിഴവുകൾ പിടിച്ച് ക്രമശീലമായി കൈകാര്യം ചെയ്യുക
   - വീണ്ടും ശ്രമിക്കൽ ലജിക് നടപ്പിലാക്കുക
   - വ്യത്യസ്ത നടപ്പിലാക്കലിലേക്ക് ഫാല്ബാക്ക് ചെയ്യുക

6. **ലോഗിംഗും നിരീക്ഷണവും**
   - ഫംഗ്ഷൻ പ്രവർത്തന സമയം ട്രാക്ക് ചെയ്യുക
   - പാരാമീറ്ററുകളും ഫലങ്ങളും ലോഗ് ചെയ്യുക
   - ഉപയോഗ പാറ്റേണുകൾ നിരീക്ഷിക്കുക

### 🔑 ഡെക്കറേറ്ററുകളിൽ നിന്നുള്ള പ്രധാന വ്യത്യാസങ്ങൾ:

| ഫീച്ചർ | ഡെക്കറേറ്റർ | മിഡിൽവെയർ |
|---------|-----------|------------|
| **പരിധി** | ഒറ്റ ഫംഗ്ഷൻ | ഏജന്റിലെ എല്ലാ ഫംഗ്ഷനുകളും |
| **സൗകര്യം** | നിർവചന സമയത്ത് ഉറപ്പിക്കപ്പെട്ടത് | റൺടൈം-ൽ ഡൈനാമിക് |
| **സന്ധർഭം** | പരിമിതം | മുഴുവൻ ഏജന്റ് സന്ധർഭം |
| **സമാകലനം** | പല ഡെക്കറേറ്ററുകളും | മിഡിൽവെയർ പൈപ്പ്‌ലൈൻ |
| **ഏജന്റ്-അറിവുള്ളത്** | ഇല്ല | ഉണ്ട് (ഏജന്റ് സ്ഥിതിക്ക് ആക്‌സസ്) |

### 📚 എപ്പോൾ മിഡിൽവെയർ ഉപയോഗിക്കണം:

✅ **മിഡിൽവെയർ ഉപയോഗിക്കുക:**
- ഉപയോക്താവിന്റെയും സെഷൻ നിലയുടെയും അടിസ്ഥാനത്തിൽ പെരുമാറ്റം മാറ്റേണ്ടതാണ്
- നിങ്ങൾക്ക് പല ഫംഗ്ഷനുകൾക്കും തർക്കം പ്രയോഗിക്കുകയായിരുന്നു
- ഏജന്റ്-നിബന്ധിതമായ സന്ധർഭം ആക്‌സസ് വേണ്ടതാണ്
- നിങ്ങൾ ക്രോസ്-കട്ടിംഗ് ആശങ്കകൾ (ലോഗിംഗ്, ഓതെന്റ് വികേഷൻ, മുതലായവ) നടപ്പിലാക്കുകയാണ്

❌ **മിഡിൽവെയർ ഉപയോഗിക്കേണ്ടതില്ല:**
- ലളിതമായ ഇൻപുട്ട് സാധൂകരിക്കൽ (പൈഡാന്റിക് ഉപയോഗിക്കുക)
- ഫംഗ്ഷൻ-സവിശേഷ തർക്കങ്ങൾ (ഫംഗ്ഷനിൽ സൂക്ഷിക്കുക)
- ഒരു തവണ വരെയുള്ള മാറ്റങ്ങൾ (വെറും ഫംഗ്ഷൻ മാറ്റുക)

### 🎓 അഭിഭാഷകന് മാതൃകകൾ:

```python
# ബഹുസംഖ്യ മിഡിൽവെയർ (പ്രവർത്തനാനുക്രമം പ്രധാനമാണ്!)
middleware=[
    logging_middleware,      # ലോഗുകൾ ആദ്യം
    auth_middleware,         # തുടർന്ന് അഥന്റിക്കേഷൻ പരിശോധിക്കുന്നു
    cache_middleware,        # തുടർന്ന് കാഷെ പരിശോധിക്കുന്നു
    rate_limit_middleware,   # തുടർന്ന് നിരക്ക് പരിധികള്‍
    priority_check_middleware  # ഒടുവിൽ പ്രാധാന്യ പരിശോധന
]

# നിബന്ധനാപൂർവം മിഡിൽവെയർ പ്രവർത്തനം
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ഫലം തിരുത്തുക
    else:
        # പ്രവർത്തനം പൂർണ്ണമായും ഒഴിവാക്കുക
        context.result = cached_value
```

### 🔗 ബന്ധപ്പെട്ട ആശയങ്ങൾ:

- **ഏജന്റ് മിഡിൽവെയർ**: agent.run() കോളുകൾ ഇടപെടുന്നു
- **ഫംഗ്ഷൻ മിഡിൽവെയർ**: ടൂൾ ഫംഗ്ഷൻ കോളുകൾ ഇടപെടുന്നു (നാം ഉപയോഗിച്ചത്!)
- **മിഡിൽവെയർ പൈപ്പ്‌ലൈൻ**: ക്രമത്തിൽ മിഡിൽവെയർ ചെയിൻ പ്രവർത്തിക്കുന്നു
- **സന്ധർഭ പ്രചരണം**: മിഡിൽവെയർ ചെയിനിലൂടെ സ്ഥിതിവിവരം പാസ്സ് ചെയ്യുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
